# Agents Evaluation Framework

## Overview
This notebook demonstrates comprehensive techniques for evaluating Agent Performance, Tool Execution and Reliability

### Agents Evaluation Dimensions:
- **Agent Performance**: Measuring accuracy using ground truth datasets
- **Tool Execution**: Analyzing tool selection and execution success rates
- **Resource Efficiency**: Token usage, latency, and cycle duration analysis
- **Agent Reliability**: Consistency across multiple test scenarios

## 1. Dependencies Installation

In [1]:
!pip install strands-agents strands-agents-tools --upgrade
!pip install ddgs
!pip install bedrock_agentcore

Looking in indexes: https://pypi.org/simple, https://plugin.us-east-1.prod.workshops.aws
  Using cached strands_agents-1.50.2-py3-none-any.whl.metadata (19 kB)
Using cached strands_agents-1.50.2-py3-none-any.whl (638 kB)
  Attempting uninstall: strands-agents
    Found existing installation: strands-agents 1.50.0
    Uninstalling strands-agents-1.50.0:
      Successfully uninstalled strands-agents-1.50.0
Looking in indexes: https://pypi.org/simple, https://plugin.us-east-1.prod.workshops.aws
Looking in indexes: https://pypi.org/simple, https://plugin.us-east-1.prod.workshops.aws


## 2. Imports

In [2]:
from strands import Agent, tool 
from strands.models import BedrockModel 
from bs4 import BeautifulSoup 
import requests 
import pandas as pd, re

# Model IDs are centralised in ../model_config.py
import sys
sys.path.append("..")
from model_config import (
    DEFAULT_MODEL_ID, JUDGE_MODEL_ID, CHATBOT_MODEL_ID,
    MODEL_NAMES, QUICK_COMPARISON_MODELS,
)

## 3. Ground Truth Dataset Loading

In [3]:
# Read the CSV file
#this contains the city, state, population, and land area in square miles in 2024.
gold_standard_city_pop = pd.read_csv('city_pop.csv')
# Clean the dataset once when loading, wikipedia has commas in the numbers.
gold_standard_city_pop['population'] = gold_standard_city_pop['population'].astype(str).str.replace(',', '').astype(float)
gold_standard_city_pop['land_area_mi2'] = gold_standard_city_pop['land_area_mi2'].astype(str).str.replace(',', '').astype(float)

# Show the first 3 rows, as a reference
print(gold_standard_city_pop.head(3))  # First 3 rows

          city state  population  land_area_mi2
0  New York[c]    NY   8478072.0          300.5
1  Los Angeles    CA   3878704.0          469.5
2      Chicago    IL   2721308.0          227.7


## 4. Core Evaluation Function

This function performs comprehensive evaluation of agent responses by extracting structured data from XML tags and calculating accuracy metrics. It parses population and area estimates from the agent's response using regex patterns,then compares these against ground truth data from the dataset to compute percent error rates. 

The function also captures key performance metrics including token usage, execution time, and tool call frequency from the agent's built-in observability system. This dual approach enables both accuracy assessment and resource efficiency analysis in a single evaluation pass.

In [4]:
def evaluate_city_guess(city, state, chatbot_response, dataset):
    """
    Evaluate population and area guesses against the gold standard dataset.
    
    Parameters:
    - city: str, city name
    - state: str, state abbreviation (e.g., 'NY', 'CA')
    - chatbot_response: Strands AgentResult object to be evaluated
    - dataset: pandas DataFrame, the gold standard dataset
    
    Returns:
    - dict with percent errors for population and area, and total tokens, execution time, and tool calls.
    
    Raises:
    - ValueError if city/state combination not found
    """
    
    # Clean the city name for matching
    city_clean = city.strip()

    
    #use regex to grab the final answer as numbers
    final_msg = chatbot_response.message['content'][0]['text']
    try:
        guessed_pop, guessed_area = int(re.search(r'<pop>(.*?)</pop>', final_msg).group(1)), float(re.search(r'<area>(.*?)</area>', final_msg).group(1))
    except:
        raise ValueError(f"XML tags not found in reply")

    
    #extract agent loop metrics
    total_tokens = chatbot_response.metrics.accumulated_usage['totalTokens']
    total_time = sum(chatbot_response.metrics.cycle_durations)
    
    tool_calls = 0
    for t in chatbot_response.metrics.tool_metrics.keys():
        tool_calls+= chatbot_response.metrics.tool_metrics[t].call_count

    
    # Find the city in the dataset
    # Use case-insensitive matching and handle potential annotations
    mask = (dataset['city'].str.replace(r'\[.*\]', '', regex=True).str.strip().str.lower() == city_clean.lower()) & \
           (dataset['state'].str.upper() == state.upper())
    
    matching_rows = dataset[mask]
    
    if len(matching_rows) == 0:
        raise ValueError(f"City '{city}' in state '{state}' not found in dataset")
    
    if len(matching_rows) > 1:
        print(f"Warning: Multiple matches found for {city}, {state}. Using first match.")
    
    # Get the actual values
    actual_pop = matching_rows.iloc[0]['population']
    actual_area = matching_rows.iloc[0]['land_area_mi2']
    
    # Calculate percent error: |actual - guess| / actual * 100
    pop_error = abs(actual_pop - guessed_pop) / actual_pop * 100
    area_error = abs(actual_area - guessed_area) / actual_area * 100
    
    return {
        'city': matching_rows.iloc[0]['city'],
        'state': matching_rows.iloc[0]['state'],
        'actual_population': actual_pop,
        'guessed_population': guessed_pop,
        'population_error_percent': round(pop_error, 2),
        'actual_area': actual_area,
        'guessed_area': guessed_area,
        'area_error_percent': round(area_error, 2),
        'total_tokens': total_tokens,
        'total_time': total_time,
        'tool_calls': tool_calls
    }

## 5. Agent Tools

These two tools enable web-based information retrieval for the agent. 

* The web_search tool queries DDGS (a metasearch library) to find relevant web pages, returning up to 5 results with titles, URLs, and snippets.
* The get_page tool extracts raw text content from specific URLs, allowing the agent to dive deeper into search results.

Together, they provide a complete web research capability that enables agents to access current information beyond their training data when evaluating tasks like population queries.

In [5]:
@tool
def web_search(topic: str) -> str:
    """Search the web for a given topic using DDGS metasearch."""
    import subprocess, json, sys
    try:
        result = subprocess.run(
            [sys.executable, '-c',
             'import json; from ddgs import DDGS; '
             f'r=DDGS(timeout=4).text({json.dumps(topic)}, max_results=3, backend="html"); '
             'print(json.dumps(r))'],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode != 0:
            return f"Search error: {result.stderr.strip()}"
        results = json.loads(result.stdout)
        if not results:
            return "No search results found"
        result_string = ""
        for i, r in enumerate(results):
            result_string += f"Result {i+1}: {r.get('title', 'No title')}\nURL: {r.get('href', 'No URL')}\nSnippet: {r.get('body', 'No description')}\n\n"
        return result_string
    except subprocess.TimeoutExpired:
        return "Search timed out after 10 seconds"
    except Exception as e:
        return f"Search error: {str(e)}"
    
@tool      
def get_page(url: str) -> str:
    """this function takes a URL and returns the raw text from that page.
    it can be used to get more info based on a Google search result listing."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        bs = BeautifulSoup(response.text,'html.parser')
        return bs.text
    except Exception as e:
        return f"Error fetching page: {str(e)}"

## 6. AWS Bedrock Configuration

These configurations optimize Bedrock API calls for different evaluation scenarios. The quick_config prioritizes speed with aggressive timeouts (5s connect, 20s read) and no retries, ideal for rapid evaluation runs where fast response times are critical. The longer_config provides more tolerance (10s connect, 60s read, 1 retry) for complex queries that may require additional processing time. 

This dual approach allows the evaluation framework to balance speed and reliability based on the specific testing requirements.

In [6]:
from botocore.config import Config

#A custom config for Bedrock to only allow short connections - for our demo we expect all calls to be fast.
#here we turn off retries, and we time out after 20 seconds.
quick_config = Config(
    connect_timeout=5,
    read_timeout=20,
    retries={"max_attempts": 0}
)

longer_config = Config(
    connect_timeout=10,
    read_timeout=60,
    retries={"max_attempts": 1}
)

## 7. Single Agent Baseline Test

This establishes a baseline evaluation by creating an agent with Nova Micro (optimized for speed and cost) and testing it against a specific query. The agent is equipped with web search capabilities and configured with aggressive timeouts for rapid evaluation. 

The test demonstrates the complete evaluation pipeline: the agent searches for New York population and area data, formats the response with XML tags for automated parsing, then the evaluation function measures accuracy against ground truth data while capturing performance metrics. This baseline provides a reference point for comparing different models, configurations, and evaluation scenarios.

### Verbose Callback Handler

Before creating the agent, we define a custom callback handler to display tool calls with their parameters. This helps us understand what the agent is doing during execution.

In [7]:
import json

# Custom callback handler to display tool calls with their parameters.
# 
# Strands agents stream output via callbacks. The callback receives kwargs including:
# - 'data': streaming text chunks from the model's response
# - 'current_tool_use': info about a tool being called (name, toolUseId, input)
#
# The challenge: tool input arrives as a streaming JSON string, so the callback fires
# multiple times as each character arrives. We detect completion by attempting to parse
# the JSON - if it parses successfully, the input is complete.
#
# We track toolUseId (unique per invocation) in a set to avoid duplicate prints.
# This allows the same tool to be called multiple times - each gets a unique ID.

seen_complete_tools = set()

def verbose_callback(**kwargs):
    """Callback handler that displays streaming text and tool calls with parameters."""
    # Stream text output as it arrives
    if 'data' in kwargs:
        print(kwargs['data'], end='', flush=True)
    
    # Display tool calls when their input is complete
    if 'current_tool_use' in kwargs:
        tool_use = kwargs['current_tool_use']
        tool_id = tool_use.get('toolUseId', 'no-id')
        tool_name = tool_use.get('name', 'no-name')
        tool_input = tool_use.get('input', '')
        
        # Skip if we've already printed this tool call
        if tool_id in seen_complete_tools:
            return
            
        # Try to parse the input as JSON - if successful, it's complete
        if isinstance(tool_input, str) and tool_input:
            try:
                parsed_input = json.loads(tool_input)
                seen_complete_tools.add(tool_id)
                print(f"\n\nTool Call: {tool_name} {json.dumps(parsed_input, indent=2)}")
                print()
            except json.JSONDecodeError:
                pass  # Input is still streaming, not yet valid JSON

In [8]:
#Create the chatbot.  We'll use Nova Micro to optimize for latency, cost, and capacity
chatbot_model_name = CHATBOT_MODEL_ID
#add custom timeout for the model, to keep the tool from hanging or retrying too much.
chatbot_model = BedrockModel(
    model_id=chatbot_model_name,
    boto_client_config=quick_config    
)
# Reset the tool tracking set for a fresh run
seen_complete_tools.clear()

chatbot = Agent(tools=[web_search,get_page], model=chatbot_model, callback_handler=verbose_callback)
#Call the chat bot with a simple request.
prompt = """How many people live in New York, and what's the area of the city in square miles?
After you respond, also include your answer in 'pop' and 'area' XML tags, for programatic processing.
The values in the XML tags should only be numbers, no words or commas."""
chatbot_response = chatbot(prompt)

<thinking> To find out how many people live in New York and the area of the city in square miles, I will first need to conduct a web search for this information. Once I have the information, I will format it according to the user's request, including it in 'pop' and 'area' XML tags.</thinking>



Tool Call: web_search {
  "topic": "population of New York City"
}



Tool Call: web_search {
  "topic": "area of New York City in square miles"
}

According to the web search results, the population of New York City in July 2025 is estimated to be 8,584,629, and its area is approximately 300.46 square miles.

Here is the information formatted in XML tags as requested:

```xml
<pop>8584629</pop>
<area>300.46</area>
```

Please note that the population and area information are estimates and may vary.

In [9]:
result = evaluate_city_guess("New York", "NY", chatbot_response, gold_standard_city_pop)
print(f"Population error: {result['population_error_percent']}%")
print(f"Area error: {result['area_error_percent']}%")
print(f"Total Tokens: {result['total_tokens']} tokens")
print(f"Total Time: {result['total_time']:.2f} seconds")
print(f"Tool Calls: {result['tool_calls']}")

Population error: 1.26%
Area error: 0.01%
Total Tokens: 2639 tokens
Total Time: 5.17 seconds
Tool Calls: 2


## 8. Single Model Evaluation Tool

This function standardizes model evaluation by encapsulating the complete testing workflow into a reusable tool. It creates an agent with any specified Bedrock model, executes a consistent population query, and returns formatted performance metrics as a string. 

The function suppresses verbose output during execution and provides a clean summary of accuracy and efficiency metrics. This modular approach enables systematic comparison across different models using identical test conditions and evaluation criteria.


In [10]:
@tool
def eval_model(model_name: str) -> str:
    """Start an evaluator for a particular model.
    model_name is the model endpoint to be evaluated.
    Retruns a string containing information about this model.
    """
    #add custom timeout for the model, to keep the tool from hanging or retrying too much.
    chatbot_model = BedrockModel(
        model_id=model_name,
        boto_client_config=quick_config    
    )
    
    chatbot = Agent(tools=[web_search,get_page], model=chatbot_model, callback_handler=None)# callback_handler=None to suppress sub agent print outs
    #Call the chat bot with a simple request.
    prompt = """How many people live in Phoenix, AZ, and what's the area of the city in square miles?
    After you respond, also include your answer in 'pop' and 'area' XML tags, for programatic processing.
    The values in the XML tags should only be numbers, no words or commas."""
    chatbot_response = chatbot(prompt)
    result = evaluate_city_guess("Phoenix", "AZ", chatbot_response, gold_standard_city_pop)
    result_string = ""
    result_string = result_string + f"Population error: {result['population_error_percent']}%" + '\n'
    result_string = result_string + f"Area error: {result['area_error_percent']}%" + '\n'
    result_string = result_string + f"Total Tokens: {result['total_tokens']} tokens" + '\n'
    result_string = result_string + f"Total Time: {result['total_time']:.2f} seconds" + '\n'
    result_string = result_string + f"Tool Calls: {result['tool_calls']}"
    print (result_string)
    return result_string

## 9. Multi-Model Comparison

This demonstrates automated multi-model comparison using an agent equipped with the evaluation tool. The evaluator agent systematically tests five different models (Nova Micro, Lite, Pro, and Claude Haiku, Sonnet) using identical prompts and conditions. 

It handles failures with automatic retry logic and compiles results into a comparative table 
showing accuracy metrics, resource usage, and reliability statistics. This meta-evaluation approach showcases how agents can orchestrate complex evaluation workflows, providing comprehensive model performance analysis across the AWS Bedrock model family.


In [11]:
model_list = "\n".join(f'{name}: "{model_id}",' for model_id, name in MODEL_NAMES.items())
evaluator_prompt = f"""
Use the eval_model tool to evaluate these models:
{model_list}
Provide a table comparason on the results, and include columns for all evaluation data points, including number of tool calls, and the number of times the model failed to evaluate and had to be retried.
Do not include the endpoint names in the table, only the model names, to save space.
If a model fails to evaluate, you should retry it up to 3 times.
"""

In [12]:
evaluator = Agent(tools=[eval_model], model=chatbot_model)
evaluator_response = evaluator(evaluator_prompt)

<thinking> I need to evaluate the specified models using the `eval_model` tool. The evaluation should include the number of tool calls, and the number of times the model failed to evaluate and had to be retried. I will retry any failed evaluations up to 3 times. Here is the plan:
1. Evaluate each model endpoint using `eval_model`.
2. Collect the evaluation results.
3. Create a table with the model names and the evaluation data points.
</thinking>


Tool #1: eval_model

Tool #2: eval_model

Tool #3: eval_model

Tool #4: eval_model

Tool #5: eval_model
Population error: 2.89%
Area error: 0.02%
Total Tokens: 2079 tokens
Total Time: 4.94 seconds
Tool Calls: 2
Population error: 3.89%
Area error: 0.02%
Total Tokens: 1948 tokens
Total Time: 8.25 seconds
Tool Calls: 1
Population error: 0.29%
Area error: 0.06%
Total Tokens: 4284 tokens
Total Time: 8.73 seconds
Tool Calls: 3
Population error: 0.0%
Area error: 0.0%
Total Tokens: 7988 tokens
Total Time: 9.92 seconds
Tool Calls: 3
Population error:

## 10. Multi-City Evaluation Framework

### Expanding Evaluation to Multiple Data Points

Next, we'll expand our evaluator to be able to check based on more than one data point. We add the calculator too to assist.

In [13]:
import random


@tool
def calculate(expression: str) -> str:
    """Evaluate mathematical expressions safely. Use for calculations like population density."""
    try:
        allowed_chars = set('0123456789+-*/()., ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters"
        return str(eval(expression))
    except:
        return "Error: Invalid calculation"

### Multi-City Evaluation Function

In [14]:
import statistics
import random

def evaluate_multiple_cities(model_name, num_cities=3):
    """Multi-city evaluation using original evaluate_city_guess function"""
    MAJOR_CITIES = [
        ("New York", "NY"), ("Los Angeles", "CA"), ("Chicago", "IL"),
        ("Houston", "TX"), ("Phoenix", "AZ"), ("Philadelphia", "PA")
    ]
    
    test_cities = random.sample(MAJOR_CITIES, num_cities)
    results = []
    
    for city, state in test_cities:
        try:
            chatbot_model = BedrockModel(model_id=model_name, boto_client_config=quick_config)
            chatbot = Agent(tools=[web_search, get_page, calculate], model=chatbot_model, callback_handler=None)
            
            prompt = f"""How many people live in {city}, {state}, and what's the area of the city in square miles?
After you respond, also include your answer in 'pop' and 'area' XML tags, for programatic processing.
The values in the XML tags should only be numbers, no words or commas."""
            
            response = chatbot(prompt)
            result = evaluate_city_guess(city, state, response, gold_standard_city_pop)
            results.append(result)
            print(f"✓ {city}, {state}")
            
        except Exception as e:
            print(f"✗ Failed {city}, {state}: {e}")
            continue
    
    if results:
        return {
            'cities_tested': len(results),
            'avg_population_error': round(statistics.mean([r['population_error_percent'] for r in results]), 2),
            'avg_area_error': round(statistics.mean([r['area_error_percent'] for r in results]), 2),
            'total_tokens': sum([r['total_tokens'] for r in results]),
            'avg_time_per_city': round(statistics.mean([r['total_time'] for r in results]), 2),
            'total_tool_calls': sum([r['tool_calls'] for r in results]),
            'individual_results': results
        }
    return None

### Multi-City Evaluation Tool

In [15]:
@tool
def eval_model_multi(model_name: str, num_cities: int = 3) -> str:
    """Multi-city version of eval_model using existing evaluation logic"""
    results = evaluate_multiple_cities(model_name, num_cities)
    
    if results:
        result_string = f"Cities tested: {results['cities_tested']}\n"
        result_string += f"Avg population error: {results['avg_population_error']}%\n"
        result_string += f"Avg area error: {results['avg_area_error']}%\n"
        result_string += f"Total tokens: {results['total_tokens']}\n"
        result_string += f"Avg time per city: {results['avg_time_per_city']:.2f} seconds\n"
        result_string += f"Total tool calls: {results['total_tool_calls']}"
        print(result_string)
        return result_string
    else:
        return "Evaluation failed - no cities successfully processed"

### Test Multi-City Evaluation

In [16]:
# Test it
results = evaluate_multiple_cities(CHATBOT_MODEL_ID, 3)
if results:
    print(f"Cities tested: {results['cities_tested']}")
    print(f"Avg population error: {results['avg_population_error']}%")
    print(f"Avg area error: {results['avg_area_error']}%")

✓ New York, NY
✓ Los Angeles, CA
✓ Phoenix, AZ
Cities tested: 3
Avg population error: 0.44%
Avg area error: 9.79%


In [17]:
multi_evaluator = Agent(tools=[eval_model_multi], model=chatbot_model)
multi_model_list = "\n".join(f'- "{m}"' for m in QUICK_COMPARISON_MODELS)
multi_prompt = f"""
Use eval_model_multi to test these models on 3 cities each:
{multi_model_list}
Create a comparison table with all metrics.
"""
multi_response = multi_evaluator(multi_prompt)

<thinking> I need to use the `eval_model_multi` tool to test the provided models on 3 cities each. I will need to call this tool twice, once for each model, with `model_name` and `num_cities` parameters set to their respective values. I will then compile the results into a comparison table.</thinking>

Tool #1: eval_model_multi

Tool #2: eval_model_multi
✓ New York, NY
✓ New York, NY
✓ Houston, TX
✓ Los Angeles, CA
✓ Los Angeles, CA
Cities tested: 3
Avg population error: 0.62%
Avg area error: 3529.18%
Total tokens: 29156
Avg time per city: 7.40 seconds
Total tool calls: 12
✓ Houston, TX
Cities tested: 3
Avg population error: 2.71%
Avg area error: 20.0%
Total tokens: 14716
Avg time per city: 8.92 seconds
Total tool calls: 8
<thinking> I have received the results for both models from the `eval_model_multi` tool. I will now compile these results into a comparison table to make it easier to compare the performance metrics of both models across 3 cities each.</thinking>

Here is the compari

## 11. Tool Call Evaluation Framework

### Tool Selection Accuracy Testing

This framework evaluates agent tool selection accuracy using a structured test dataset with 20 diverse scenarios. Each test case specifies the expected tool (calculator, file operations, code interpreter, or none) for different 
query types. 

The evaluation creates an agent with multiple tools, processes each test case, and tracks which tools were actually invoked using the built-in metrics system. By comparing expected versus actual tool usage, it calculates an overall tool selection accuracy percentage, providing quantitative assessment of the agent's ability to choose appropriate tools for different task types.

In [18]:
import json

dataset = [
  { "id": 1, "input": "What is 234 + 876?", "expected_tool": "calculator", "expected_output": "1110" },
  { "id": 2, "input": "Multiply 45 by 19.", "expected_tool": "calculator", "expected_output": "855" },
  { "id": 3, "input": "What is (15 * 4) + 9?", "expected_tool": "calculator", "expected_output": "69" },
  { "id": 4, "input": "Read the contents of notes.txt", "expected_tool": "file_read", "expected_output": "File contents of notes.txt" },
  { "id": 5, "input": "Open and show me what's inside data.csv", "expected_tool": "file_read", "expected_output": "CSV content from data.csv" },
  { "id": 6, "input": "Display everything in todo.md", "expected_tool": "file_read", "expected_output": "Markdown content of todo.md" },
  { "id": 7, "input": "Write 'Hello World' into hello.txt", "expected_tool": "file_write", "expected_output": "File hello.txt created with 'Hello World'" },
  { "id": 8, "input": "Save the text 'AgentCore Rocks!' into core.txt", "expected_tool": "file_write", "expected_output": "File core.txt created with text" },
  { "id": 9, "input": "Create a file log.txt that contains 'run successful'", "expected_tool": "file_write", "expected_output": "File log.txt written" },
  { "id": 10, "input": "Run Python code: print(2+3)", "expected_tool": "code_interpreter", "expected_output": "5" },
  { "id": 11, "input": "Execute Python code: for i in range(3): print(i)", "expected_tool": "code_interpreter", "expected_output": "0\n1\n2" },
  { "id": 12, "input": "Run a Python snippet to calculate factorial of 5", "expected_tool": "code_interpreter", "expected_output": "120" },
  { "id": 13, "input": "What is the capital of France?", "expected_tool": "none", "expected_output": "Paris" },
  { "id": 14, "input": "Who is the CEO of Amazon?", "expected_tool": "none", "expected_output": "Andy Jassy" },
  { "id": 15, "input": "Divide 500 by 25.", "expected_tool": "calculator", "expected_output": "20" },
  { "id": 16, "input": "Square root of 144?", "expected_tool": "calculator", "expected_output": "12" },
  { "id": 17, "input": "Show me what's inside config.yaml", "expected_tool": "file_read", "expected_output": "YAML file content" },
  { "id": 18, "input": "Write 'Done for today' in status.txt", "expected_tool": "file_write", "expected_output": "status.txt written" },
  { "id": 19, "input": "Execute Python: sum([10,20,30])", "expected_tool": "code_interpreter", "expected_output": "60" },
  { "id": 20, "input": "What is 99 * 99?", "expected_tool": "calculator", "expected_output": "9801" }
]

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [19]:
from strands import Agent
from strands_tools import calculator, file_read, current_time, file_write
from strands_tools.code_interpreter.agent_core_code_interpreter import AgentCoreCodeInterpreter
code_interpreter = AgentCoreCodeInterpreter()

import os
os.environ['BYPASS_TOOL_CONSENT'] = 'true'

# Create agent with multiple tools
agent = Agent(
    model=DEFAULT_MODEL_ID,
    tools=[calculator, file_read, current_time, file_write, code_interpreter],
    record_direct_tool_call = True
)

# Define tool-specific test cases

# Track tool usage
tool_usage_results = []
for case in dataset:
    response = agent(case["input"])

    # Extract used tools from the response metrics
    used_tools = []
    if hasattr(response, 'metrics') and hasattr(response.metrics, 'tool_metrics'):
        for tool_name, tool_metric in response.metrics.tool_metrics.items():
            if tool_metric.call_count > 0:
                used_tools.append(tool_name)

    tool_usage_results.append({
        "query": case["input"],
        "expected_tool": case["expected_tool"],
        "used_tools": used_tools,
        "correct_tool_used": case["expected_tool"] in used_tools
    })

# Analyze tool usage accuracy
correct_usage_count = sum(1 for result in tool_usage_results if result["correct_tool_used"])
accuracy = correct_usage_count / len(tool_usage_results)
print('\n Results:\n')
print(f"Tool selection accuracy: {accuracy:.2%}")

tool=<<strands_tools.code_interpreter.agent_core_code_interpreter.AgentCoreCodeInterpreter object at 0x119cadfd0>> | unrecognized tool specification



Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 234 + 876           │                                                                            │
│  │ Result    │ 1110                │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

234 + 876 = **1110**
Tool #2: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 45 * 19             │                                                                            │
│  │ Result    │ 855                 │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

I've written "Done for today" to **status.txt** successfully.
Tool #16: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 10+20+30            │                                                                            │
│  │ Result    │ 60                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I don't have a Python execution environment, but I calculated the equivalent result:

`sum([10, 20, 30])` = **60**
Tool #17: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 99*99               │                                                                            │
│  │ Result    │ 9801                │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**99 × 99 = 9801**
 Results:

Tool selection accuracy: 70.00%


## 12. LLM As a Judge Evaluation
### Using Stronger Models to Evaluate Agent Responses
This implements LLM-as-a-Judge methodology by using a stronger model to evaluate responses from a weaker model. The evaluator agent judges each criterion — accuracy, relevance, completeness, and tool usage — as a **binary pass/fail check**, plus an overall verdict that passes only if every check passes. For each test case, it compares the agent's response against expected outputs and provides detailed qualitative feedback. 

We deliberately avoid 1-5 rating scales: they drift between runs, invite middle-value hedging, and are harder to act on than "failed the *accuracy* check". Granularity comes from the four specific checks and their pass rates (see `02-quality-metrics/03_Evaluating_your_Judge.ipynb` for the full reasoning).

This approach enables scalable evaluation of subjective response quality aspects that are difficult to measure with traditional metrics, while leveraging the superior reasoning capabilities of advanced models for consistent assessment.

In [20]:
from strands import Agent
import json
# Create the agent to evaluate
# agent = Agent(model=DEFAULT_MODEL_ID)
# Create an evaluator agent with a stronger model
evaluator = Agent(
    model=JUDGE_MODEL_ID,
    system_prompt="""
    You are an expert AI evaluator. Assess the quality of AI responses as a BINARY
    pass/fail verdict on each of these checks. A check PASSES only if it fully meets
    the criterion — when in doubt, fail it. Do not use rating scales.
    1. Accuracy - PASS if every factual claim in the response is correct; FAIL if any is wrong or unsupported.
    2. Relevance - PASS if the response directly addresses the query; FAIL otherwise.
    3. Completeness - PASS if all aspects of the query are addressed; FAIL if any part is missing.
    4. Tool usage - PASS if the right tools were used appropriately; FAIL if tools were wrong, missing, or unnecessary.
    Give a verdict (PASS or FAIL) with a brief reason for each check, then an overall
    verdict that is PASS only if all four checks pass.
    """
)
# Load test cases
with open("dataset.json", "r") as f:
    test_cases = json.load(f)
# Run evaluations
evaluation_results = []
for case in test_cases:
    # Get agent response
    print(case)
    agent_response = agent(case['input'])
    # Create evaluation prompt
    eval_prompt = f"""
    Query: {case['input']}
    Response to evaluate:
    {agent_response}
    Expected response (if available):
    {case.get('expected_output', 'Not provided')}
    Please evaluate the response with a pass/fail verdict on accuracy, relevance, completeness, and tool usage, plus the overall verdict.
    """
    # Get evaluation
    evaluation = evaluator(eval_prompt)
    # Store results
    evaluation_results.append({
        "test_id": case.get("id", ""),
        "query": case["input"],
        "agent_response": str(agent_response),
        "evaluation": evaluation.message['content']
    })
# Save evaluation results
with open("evaluation_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)

{'id': 1, 'input': 'What is 234 + 876?', 'expected_tool': 'calculator', 'expected_output': '1110'}

Tool #18: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 234+876             │                                                                            │
│  │ Result    │ 1110                │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**234 + 876 = 1110**## Evaluation

**1. Accuracy: PASS**
234 + 876 = 1110 is mathematically correct.

**2. Relevance: PASS**
The response directly answers the arithmetic question asked.

**3. Completeness: PASS**
The query only asked for a single sum, and the response provides the complete answer with no missing components.

**4. Tool usage: PASS**
This is a simple arithmetic calculation that does not require external tools; no tool usage was necessary, and none was inappropriately used or omitted.

## Overall Verdict: **PASS**
All four criteria are satisfied — the response is accurate, relevant, complete, and appropriately handled without unnecessary tool use.{'id': 2, 'input': 'Multiply 45 by 19.', 'expected_tool': 'calculator', 'expected_output': '855'}

Tool #19: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 45*19               │                                                                            │
│  │ Result    │ 855                 │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**45 × 19 = 855**## Evaluation

**1. Accuracy: PASS**
45 × 19 = 855 is mathematically correct.

**2. Relevance: PASS**
The response directly addresses the multiplication query asked.

**3. Completeness: PASS**
The query asked for a single product, and the response provides the complete answer with no missing components.

**4. Tool usage: PASS**
This is a simple arithmetic calculation that does not require external tools; no tool usage was necessary, and none was inappropriately used or omitted.

## Overall Verdict: **PASS**
All four criteria are satisfied — the response is accurate, relevant, complete, and appropriately handled without unnecessary tool use.{'id': 3, 'input': 'What is (15 * 4) + 9?', 'expected_tool': 'calculator', 'expected_output': '69'}

Tool #20: calculator


...

I've written "Done for today" to **status.txt** successfully.## Evaluation

**1. Accuracy: FAIL**
The response claims the write operation was completed successfully, but there is no evidence that a file-writing tool was actually invoked. Without a demonstrated tool call or system confirmation output, the claim of success cannot be verified and may be a hallucinated/assumed outcome.

**2. Relevance: PASS**
The response directly addresses the query's intent — confirming that the requested text was written to the specified file.

**3. Completeness: FAIL**
The response is brief and provides no confirmation details (e.g., file path validation, tool-generated success status, or verification that status.txt now contains exactly "Done for today"). Since the underlying action is unverified, the completeness of the task cannot be substantiated.

**4. Tool usage: FAIL**
The query explicitly requires a write operation to a file, which necessitates a file-writing tool call (e.g., write_file). There

╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 10+20+30            │                                                                            │
│  │ Result    │ 60                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Result: 60**

This matches what Python's `sum([10,20,30])` would return: `60`## Evaluation

**1. Accuracy: PASS**
The sum of [10, 20, 30] is indeed 60, which is mathematically correct.

**2. Relevance: PASS**
The response addresses the query by providing the numeric result that the Python expression would produce.

**3. Completeness: FAIL**
Unlike prior similar cases where the AI transparently disclosed the absence of a Python execution tool and explained the substitute method used (e.g., calculator), this response lacks that clarity. It ambiguously states the result "matches what Python's `sum()` would return," which implies the calculation was done separately but doesn't explicitly confirm whether actual Python execution occurred, what tool was used, or whether no such tool is available. This omission leaves the response incomplete regarding tool transparency.

**4. Tool usage: FAIL**
There's no clear evidence or disclosure of what tool (if any) was actually used to compute this re

╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 99*99               │                                                                            │
│  │ Result    │ 9801                │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**99 × 99 = 9801**## Evaluation

**1. Accuracy: PASS**
99 × 99 = 9801 is mathematically correct.

**2. Relevance: PASS**
The response directly addresses the multiplication query asked.

**3. Completeness: PASS**
The query asked for a single product, and the response provides the complete answer with no missing components.

**4. Tool usage: PASS**
This is a simple arithmetic calculation that does not require external tools; no tool usage was necessary, and none was inappropriately used or omitted.

## Overall Verdict: **PASS**
All four criteria are satisfied — the response is accurate, relevant, complete, and appropriately handled without unnecessary tool use.

## 13. Advanced Metrics Analysis

### Detailed Performance Metrics Using Observability

In [21]:
result = agent("What is the square root of 144?")

def display_metrics(result):
    summary = result.metrics.get_summary()
    
    print(" Agent Performance Summary")
    print("=" * 50)
    
    # Core metrics
    print(f" Execution: {summary['total_cycles']} cycles in {summary['total_duration']:.2f}s")
    print(f" Average cycle time: {summary['average_cycle_time']:.2f}s")
    
    # Tool usage
    print(f"\n Tool Performance:")
    for tool, data in summary['tool_usage'].items():
        stats = data['execution_stats']
        print(f"   {tool}: {stats['call_count']} calls | {stats['success_rate']:.0%} success | {stats['average_time']*1000:.1f}ms avg")
    
    # Token usage
    usage = summary['accumulated_usage']
    print(f"\n Token Usage:")
    print(f"   Input: {usage['inputTokens']:,} | Output: {usage['outputTokens']:,} | Total: {usage['totalTokens']:,}")
    
    # Latency
    print(f" Total latency: {summary['accumulated_metrics']['latencyMs']:,}ms")
    
    # Cycle breakdown
    print(f"\n Cycle Details:")
    for i, trace in enumerate(summary['traces'], 10):
        if trace['duration']:
            print(f"   Cycle {i}: {trace['duration']:.2f}s")
display_metrics(result)

**√144 = 12**

(Same as calculated earlier — 12 × 12 = 144) Agent Performance Summary
 Execution: 75 cycles in 165.88s
 Average cycle time: 2.21s

 Tool Performance:
   calculator: 18 calls | 100% success | 5.9ms avg
   file_read: 8 calls | 100% success | 13.7ms avg
   file_write: 8 calls | 100% success | 8.0ms avg

 Token Usage:
   Input: 408,823 | Output: 5,074 | Total: 413,897
 Total latency: 161,634ms

 Cycle Details:
   Cycle 10: 2.17s
   Cycle 11: 1.51s
   Cycle 12: 1.98s
   Cycle 13: 1.63s
   Cycle 14: 1.95s
   Cycle 15: 1.89s
   Cycle 16: 2.01s
   Cycle 17: 1.91s
   Cycle 18: 1.84s
   Cycle 19: 2.29s
   Cycle 20: 1.97s
   Cycle 21: 2.86s
   Cycle 22: 2.27s
   Cycle 23: 1.68s
   Cycle 24: 2.34s
   Cycle 25: 2.94s
   Cycle 26: 3.36s
   Cycle 27: 1.86s
   Cycle 28: 2.96s
   Cycle 29: 1.83s
   Cycle 30: 3.82s
   Cycle 31: 2.18s
   Cycle 32: 2.57s
   Cycle 33: 2.67s
   Cycle 34: 2.85s
   Cycle 35: 2.04s
   Cycle 36: 1.70s
   Cycle 37: 1.73s
   Cycle 38: 1.69s
   Cycle 39: 1.99s
   C

## 14. Conclusions and Key Findings

### Strands Agent Evaluation Framework Summary

This comprehensive evaluation framework demonstrates multiple approaches for assessing Strands agent performance:

**🎯 Accuracy Assessment:**
- Ground truth validation using structured outputs (XML tags)
- Population and area estimation error calculations
- Multi-city consistency testing

**⚡ Performance Monitoring:**
- Strands `AgentResult.metrics` for comprehensive analysis
- Token usage tracking for cost optimization
- Execution time and cycle duration measurement
- Tool call frequency and success rate analysis

**🔧 Tool Effectiveness:**
- Tool selection accuracy across different task types
- Multi-tool coordination assessment
- Tool execution success rate monitoring

**📊 Advanced Evaluation Techniques:**
- LLM-as-a-Judge for qualitative assessment
- Batch evaluation for consistency analysis
- Comparative model performance analysis

### Key Insights:

**Model Performance Patterns:**
- Larger models (Claude, Nova Pro) show better accuracy but higher token costs
- Smaller models (Nova Micro) are faster but less reliable with complex instructions
- Tool selection accuracy varies significantly between model families

**Strands Framework Benefits:**
- Built-in observability provides comprehensive performance metrics
- Agent cycle tracking enables detailed execution analysis
- Tool metrics facilitate optimization of agent capabilities
- Structured evaluation supports production monitoring

**Evaluation Methodology Learnings:**
- Structured output requirements (XML tags) are crucial for automated evaluation
- Multi-city testing reveals consistency issues not apparent in single-case tests
- LLM-as-a-Judge provides valuable qualitative insights
- Tool call efficiency is as important as accuracy for production deployments

### Production Recommendations:

1. **Implement structured outputs** for automated evaluation pipelines
2. **Use Strands metrics** for continuous performance monitoring
3. **Establish accuracy baselines** using ground truth datasets
4. **Monitor tool success rates** for reliability assessment
5. **Track token efficiency** for cost optimization
6. **Deploy LLM-as-a-Judge** for qualitative response evaluation

### Future Enhancements:

**Expanded Test Coverage:**
- Additional domains beyond city demographics
- More complex multi-step reasoning tasks
- Real-time data accuracy validation

**Advanced Metrics:**
- Semantic similarity scoring for text outputs
- Confidence calibration analysis
- Error pattern classification

**Automation Improvements:**
- Continuous evaluation pipelines
- A/B testing frameworks
- Performance regression detection

---

**📚 Reference**: This evaluation framework follows Strands documentation best practices for agent observability and performance measurement. For more details, see: https://strandsagents.com/docs/user-guide/observability-evaluation/evaluation/

**🔗 Repository**: Save this notebook and datasets for reproducible evaluations and comparative analysis across different model versions and configurations.